# An Introduction to the Foundations and Impacts of Artificial Intelligence

### UW Allen School Admitted Students Day 2026

[**James Weichert**](https://jpw.info), Assistant Teaching Professor

**Using This Notebook**

This "notebook" combines Python code with text to guide you through the process of building a _k-nearest neighbors_ classifier, a type of machine learning (or AI) algorithm.

To run the code in this notebook, **click on each code cell and press `Shift` + `Return` on your keyboard** (or click the play button in the tool bar).

In [9]:
# Run this cell (by pressing Shift + Return) 
# to import the necessary Python libraries for this demo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
import plotly.express as px

## Seattle Restaurant Recommender &trade;

James is a **big** foodie and he loves trying local restaurants. Help James build an _algorithm_—a computer program that performs a specific task—to recommend a Seattle restaurant based on user preferences.

The `seattle_restaurants` table below has nearly 200 restaurants from across the city, along with information about their average price (in dollars), average rating (out of 5), and number of reviews. This data is sourced from [this Kaggle dataset](https://www.kaggle.com/datasets/oklena/seattle-area-restaurants) based on Yelp reviews.

In [2]:
seattle_restaurants = pd.read_csv("seattle_restaurants.csv")
seattle_restaurants[['Name', 'Avg. Price', 'Avg. Rating', 'Num. Reviews']]

,Name,Avg. Price,Avg. Rating,Num. Reviews
0,Tavolàta,36.96,4.21,343.0
1,Local 360,30.50,3.96,3163.0
2,Café Campagne,22.75,4.20,1153.0
3,Katsu-ya Seattle,34.66,4.36,89.0
4,Tamari Bar,28.91,4.24,392.0
...,...,...,...,...
189,Azuki Handmade Japanese Udon Noodles,29.86,4.46,101.0
190,Eden Hill Restaurant,88.17,4.82,238.0
191,Gold Bar,28.67,4.47,87.0
192,Star Fusion & Bar,28.15,5.00,124.0


Feel free to use the `distance` function in your implementation. The function takes in the 'x', 'y' and 'z' coordinates for two points and returns the [Euclidean distance](https://en.wikipedia.org/wiki/Euclidean_distance) between them.

In [3]:
def distance(x1, y1, x2, y2, z1, z2):
    """Return the Euclidean distance between two points (x1, y1) and (x2, y2)"""
    
    return np.sqrt((x1 - x2)**2 + (y1 - y2)**2 + (z1 - z2)**2)

### Try It Yourself: Build a Restaurant Recommender &trade;

Implement the `best_match` function function, which should make a recommendation based on the closest
Seattle restaurant to the user's `ideal_price`, `ideal_rating`, and `ideal_num_reviews` preferences. The function should return the name of the restaurant in `restaurants` that best matches the user's preferences.

In [4]:
def best_match(restaurants, ideal_price, ideal_rating, ideal_num_reviews):
    """
    Given a table `restaurants`, an `ideal_price`, an `ideal_rating`, and an `ideal_num_reviews`,
    return the name of the restaurant that is the best match
    """

    n = len(restaurants)

    smallest_dist = 1_000_000 # a very large number
    best_restaurant = "" # to keep track of the restaurant we're recommending so far...

    # WRITE YOUR CODE BELOW

    # Hint: Given an index i, you can get a row of the restaurants table
    # with the expression `restaurants.iloc[i]`

    return # return the name of the recommended restaurant

##### Implementation (Open to Reveal)

In [5]:
def best_match_implementation(restaurants, ideal_price, ideal_rating, ideal_num_reviews):
    """
    Given a table `restaurants`, an `ideal_price`, an `ideal_rating`, and an `ideal_num_reviews`,
    return the name of the restaurant that is the best match
    """

    n = len(restaurants)

    smallest_dist = 1_000_000 # a very large number
    best_restaurant = "" # to keep track of the restaurant we're recommending so far...

    # WRITE YOUR CODE BELOW

    # Loop over each index in the table
    for i in range(n):

        restaurant = restaurants.iloc[i]

        price = restaurant['Avg. Price']
        rating = restaurant['Avg. Rating']
        num_reviews = restaurant['Stars_count']

        # Find the Euclidean distance using the `distance` function
        new_dist = distance(price, ideal_price, rating, ideal_rating, num_reviews, ideal_num_reviews)

        # If the new distance is smaller than what we've seen so far,
        # update the smallest distance and keep track of the new restaurant name
        if new_dist < smallest_dist:

            smallest_dist = new_dist
            best_restaurant = restaurant['Name']

    return best_restaurant # return the name of the restaurant with the smallest distance

### Testing Our Algorithm &trade;

In [6]:
# What are your ideal values for price, rating, and number of reviews?

ideal_price = 0 # replace with your value
ideal_rating = 0 # replace with your value
ideal_num_reviews = 0 # replace with your value

best_match(seattle_restaurants, ideal_price, ideal_rating, ideal_num_reviews)

In [7]:
# This cell takes care of setting up interactive sliders to change the `price` and `rating`
# and visualizing the restaurants as points on a scatter plot

# Just run me!

price = 15
rating = 4.2
num_ratings = 1000

def plot_restaurants(restaurants, price, rating, num_ratings, square_aspect = False):

  recommended_restaurant = best_match(restaurants, price, rating, num_ratings)
  colors = restaurants['Name'] == recommended_restaurant

  fig = px.scatter(restaurants, x = 'Avg. Price', y = 'Avg. Rating', hover_name = 'Name', color = colors)
  fig.add_trace(go.Scatter(x = [price], y = [rating], name='My Preference'))

  fig.update_traces(marker = {'size': 10})
  fig.update_layout(showlegend = False, title = 'Restaurant Recommender')

  if square_aspect:
    fig.update_layout(yaxis_scaleanchor="x")

  fig.show()


def update(new_price, new_rating):
    global price, rating
    price = new_price
    rating = new_rating

In [8]:
widgets.interact(update,
                 new_price = widgets.IntSlider(min = 10, max = 50, value = price, description = 'Price ($)'),
                 new_rating = widgets.FloatSlider(min = 3.5, max = 5, value = rating, description = 'Rating'));

interactive(children=(IntSlider(value=15, description='Price ($)', max=50, min=10), FloatSlider(value=4.2, des…

In [ ]:
def initialize_centroids(restaurants, k = 3):
    """Randomly pick k restaurants to be the starting point"""

    return np.array(restaurants[['Avg. Price', 'Avg. Rating', 'Stars_count']].sample(k, replace = False))  

In [ ]:
def assign_centroids(restaurants, centroids):
    """Assign each restaurant in the `restaurants` to its closest centroid in `centroids`"""

    n = len(restaurants)

    cluster_assigments = np.zeros(n)

    # Loop over each index in the table
    for i in range(n):

        restaurant = restaurants.iloc[i]
        
        price = restaurant['Avg. Price']
        rating = restaurant['Avg. Rating']
        num_ratings = restaurant['Stars_count']

        smallest_dist = 1_000_000 # a very large number

        # Loop over each centroid
        for k in range(len(centroids)):

            centroid = centroids[k]

            new_dist = distance(price, centroid[0], rating, centroid[1], num_ratings, centroid[2])

            # If the restaurant is closer to this centroid...
            if new_dist < smallest_dist:
                
                # ...assign the restaurant to the centroid
                smallest_dist = new_dist
                cluster_assigments[i] = k

    return cluster_assigments


In [ ]:
def update_centroids(restaurants, cluster_assigments, num_clusters = 3):
    """Recalculate each centroid to be the average of the points in its cluster"""

    centroids = np.zeros((num_clusters, 3))

    # Loop over each centroid
    for k in range(num_clusters):

        # Running totals
        price_sum = 0
        rating_sum = 0
        num_ratings_sum = 0

        count = 0

        # For each restaurant in the cluster k:
        for i in range(len(restaurants)):

            if cluster_assigments[i] == k:

                # ...add its values to the running totals
                price_sum += restaurants.iloc[i]['Avg. Price']
                rating_sum += restaurants.iloc[i]['Avg. Rating']
                num_ratings_sum += restaurants.iloc[i]['Stars_count']

                count += 1

        # At the end, divide by the number of restaurants in the cluster
        # to calculate the average --> this becomes the centroid value
        if count != 0:
            centroids[k][0] = price_sum / count
            centroids[k][1] = rating_sum / count
            centroids[k][2] = num_ratings_sum / count

    return centroids


In [60]:
centroids = initialize_centroids(seattle_restaurants)

for i in range(100):

    cluster_assignments = assign_centroids(seattle_restaurants, centroids)

    centroids = update_centroids(seattle_restaurants, cluster_assignments)